# Part 1: CNN From ScratchBuild a CNN from scratch for landmark classification (50 classes).**Target accuracy: ≥50%**

In [ ]:
import torchimport torch.nn as nnimport torch.optim as optimfrom torch.optim.lr_scheduler import ReduceLROnPlateauimport matplotlib.pyplot as pltimport syssys.path.insert(0, '.')from src.data import get_data_loaders, visualize_samplesfrom src.model import MyModelfrom src.optimization import get_loss, get_optimizerfrom src.train import train_model, test_model, plot_training_historyfrom src.helpers import get_device, count_parameters

## 1. Load DataThe dataset contains images of 50 world landmarks, split into train/valid/test.

In [ ]:
DATA_DIR = "landmark_images"BATCH_SIZE = 64data_loaders, image_datasets = get_data_loaders(DATA_DIR, batch_size=BATCH_SIZE)class_names = image_datasets["train"].classesprint(f"Number of classes: {len(class_names)}")print(f"Training samples: {len(image_datasets['train'])}")print(f"Validation samples: {len(image_datasets['valid'])}")print(f"Test samples: {len(image_datasets['test'])}")

## 2. Visualize Samples

In [ ]:
visualize_samples(data_loaders["train"], class_names, n=5)

## 3. Define CNN Architecture5-layer CNN with BatchNorm, progressive channel growth (16→32→64→128→256), and Dropout.

In [ ]:
device = get_device()print(f"Using device: {device}")model = MyModel(num_classes=50, dropout=0.5).to(device)count_parameters(model)

## 4. Training Setup

In [ ]:
criterion = get_loss()optimizer = get_optimizer(model, learning_rate=0.001, weight_decay=1e-5)scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, verbose=True)

## 5. Train the Model (30 epochs)

In [ ]:
history = train_model(    model, data_loaders, optimizer, criterion, scheduler,    device=device, n_epochs=30, save_path="cnn_scratch_best.pth")

## 6. Training Curves

In [ ]:
plot_training_history(history)

## 7. Test the Model

In [ ]:
# Load best modelmodel.load_state_dict(torch.load("cnn_scratch_best.pth", map_location=device))model.to(device)test_acc = test_model(model, data_loaders["test"], device)print(f"\nTarget: ≥50% | Achieved: {test_acc:.2f}%")assert test_acc >= 50, f"Test accuracy {test_acc:.2f}% is below the 50% target!"